In [1]:
!pip install transformers datasets torch scikit-learn pandas


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [3]:
df = pd.read_csv('/content/cleaned_data.csv')
df

,id,topic,sentiment,text,tokens,clean_text
0,2401,Borderlands,Positive,i am coming to the borders and i will kill you...,"['coming', 'borders', 'kill']",coming borders kill
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you all,"['im', 'getting', 'borderlands', 'kill']",im getting borderlands kill
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,"['im', 'coming', 'borderlands', 'murder']",im coming borderlands murder
3,2401,Borderlands,Positive,im getting on borderlands and i will murder y...,"['im', 'getting', 'borderlands', 'murder']",im getting borderlands murder
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,"['im', 'getting', 'borderlands', 'murder']",im getting borderlands murder
...,...,...,...,...,...,...
74676,9200,Nvidia,Positive,just realized that the windows partition of my...,"['realized', 'windows', 'partition', 'mac', 'l...",realized windows partition mac like years behi...
74677,9200,Nvidia,Positive,just realized that my mac window partition is ...,"['realized', 'mac', 'window', 'partition', 'ye...",realized mac window partition years behind nvi...
74678,9200,Nvidia,Positive,just realized the windows partition of my mac ...,"['realized', 'windows', 'partition', 'mac', 'y...",realized windows partition mac years behind nv...
74679,9200,Nvidia,Positive,just realized between the windows partition of...,"['realized', 'windows', 'partition', 'mac', 'l...",realized windows partition mac like years behi...


In [4]:
df['clean_text'] = df['clean_text'].fillna('')
df = df[df['clean_text'].str.len() > 10]

In [5]:
label_map = {
    'Positive': 0,
    'Neutral': 1,
    'Negative': 2,
    'Irrelevant': 1  # treat as Neutral
}

df['label'] = df['sentiment'].map(label_map)

/tmp/ipykernel_1240/1284604375.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['sentiment'].map(label_map)


In [6]:
critical_words = [
    "urgent", "asap", "immediately", "critical",
    "emergency", "crash", "not working",
    "failure", "error", "server down"
]

In [7]:
def add_critical(row):
    text = row['clean_text'].lower()
    if any(word in text for word in critical_words):
        return 3
    return row['label']


df['label'] = df.apply(add_critical, axis=1)

# Check distribution
print("Label Distribution:")
print(df['label'].value_counts())

Label Distribution:
label
1    28909
2    19901
0    18320
3     1032
Name: count, dtype: int64


/tmp/ipykernel_1240/3786663110.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df.apply(add_critical, axis=1)


In [8]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['clean_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = CustomDataset(train_encodings, train_labels)
val_dataset = CustomDataset(val_encodings, val_labels)


In [11]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=4
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [14]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Custom Trainer with weighted loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [16]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',

    save_strategy="no"   # 🔥 IMPORTANT FIX
)
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
500,0.081766
1000,0.104587
1500,0.127746
2000,0.134571
2500,0.151720
3000,0.123926
3500,0.110039
4000,0.089303
4500,0.155587
5000,0.128510


TrainOutput(global_step=34085, training_loss=0.0634127302954619, metrics={'train_runtime': 6200.333, 'train_samples_per_second': 43.973, 'train_steps_per_second': 5.497, 'total_flos': 1.793430046606848e+16, 'train_loss': 0.0634127302954619, 'epoch': 5.0})

In [17]:
preds = trainer.predict(val_dataset)
pred_labels = preds.predictions.argmax(axis=1)

In [18]:
label_names = {
    0: "Positive",
    1: "Neutral",
    2: "Negative",
    3: "Critical/Urgent"
}

predicted_sentiments = [label_names[i] for i in pred_labels]

In [19]:
def get_priority_score(label):
    return {
        0: 1,
        1: 2,
        2: 3,
        3: 5
    }[label]

priority_scores = [get_priority_score(i) for i in pred_labels]

In [20]:
results_df = pd.DataFrame({
    "Text": val_texts,
    "Predicted_Label": pred_labels,
    "Sentiment": predicted_sentiments,
    "Priority_Score": priority_scores
})

print(results_df.head())

                                                Text  Predicted_Label  \
0  excited announce dave matthews close verizons ...                1   
1                   red dead redemption looking dope                0   
2  thats good one havent super ultra passionate g...                0   
3  im excited see everyone play cyberpunk complet...                0   
4       saw one run today vermont cant stop thinking                2   

  Sentiment  Priority_Score  
0   Neutral               2  
1  Positive               1  
2  Positive               1  
3  Positive               1  
4  Negative               3  


In [22]:
from sklearn.metrics import classification_report
print(classification_report(val_labels, pred_labels))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94      3648
           1       0.95      0.96      0.96      5717
           2       0.95      0.95      0.95      4063
           3       1.00      1.00      1.00       205

    accuracy                           0.95     13633
   macro avg       0.96      0.96      0.96     13633
weighted avg       0.95      0.95      0.95     13633



In [23]:
print(results_df['Predicted_Label'].value_counts())

Predicted_Label
1    5751
2    4049
0    3629
3     204
Name: count, dtype: int64
